In [1]:
import pandas as pd

df_alpha = pd.read_csv('Variantes_Para_AlphaGenome.csv', sep=',')

In [2]:
df_alpha

,Cromosoma,Posicion,REF,ALT,Gen,Cambio_Proteina,Peso_ElasticNet,Hazard_Ratio,Pacientes_con_esta_Variante,ClinSig,gnomAD
0,19,15170800,T,G,NOTCH3,p.Asn1588His,0.294927,1.343028,2,Uncertain_significance,NaN
1,10,102067252,C,G,HPS6,p.Pro593Arg,0.189975,1.209220,2,NaN,NaN
2,10,102066601,G,A,HPS6,p.Arg376His,0.189975,1.209220,1,Uncertain_significance,0.000006
3,10,102066820,C,G,HPS6,p.Ser449Cys,0.189975,1.209220,1,Uncertain_significance,0.000380
4,1,43352260,G,A,MPL,p.Arg537Gln,0.163436,1.177550,1,Uncertain_significance,0.000104
...,...,...,...,...,...,...,...,...,...,...,...
97,3,93905889,T,G,PROS1,p.Asn198His,NaN,NaN,1,Uncertain_significance,0.000053
98,1,46030204,G,A,MAST2,p.Arg847His,NaN,NaN,1,Uncertain_significance,0.000052
99,2,43824423,G,C,ABCG5,p.Thr305Arg,NaN,NaN,1,Conflicting_classifications_of_pathogenicity,0.000006
100,X,154359007,T,C,FLNA,p.Gln1484Arg,NaN,NaN,1,Conflicting_classifications_of_pathogenicity,0.001663


In [6]:
# Cargar la clave de API guardada en el archivo .env
import os
from dotenv import load_dotenv

load_dotenv() 

api_key = os.getenv("ALPHAGENOME_API_KEY")

if not api_key:
    raise ValueError("La variable ALPHAGENOME_API_KEY no se encontró.")

In [7]:
# Validación con AlphaGenome
from alphagenome.data import genome
from alphagenome.models import dna_client, variant_scorers

print(" CONSULTA ALPHAGENOME")

archivo_input = "Variantes_Para_AlphaGenome.csv"
df_variantes_a_puntuar = pd.read_csv(archivo_input)
total_variantes = len(df_variantes_a_puntuar)

model = dna_client.create(api_key)

# Modalidades clave
scorers_clinvar = [
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITES'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITE_USAGE'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_JUNCTIONS']
]

# Tejidos de interés
tejidos_diana = ['liver', 'vena cava', 'vein', 'endothelial', 'blood vessel', 'hematopoietic']

os.makedirs('resultados_alphagenome', exist_ok=True)
archivo_salida = 'resultados_alphagenome/anotacion_alphagenome.tsv'

# Borrar archivo antiguo
if os.path.exists(archivo_salida):
    os.remove(archivo_salida)

for index, row in df_variantes_a_puntuar.iterrows():
    col_pos = 'Posicion' if 'Posicion' in row else 'Posición'
    
    cromosoma_raw = str(row['Cromosoma']).strip()
    cromosoma = f"chr{cromosoma_raw}" if not cromosoma_raw.lower().startswith('chr') else cromosoma_raw
    posicion = int(row[col_pos])
    ref = str(row['REF'])
    alt = str(row['ALT'])
    
    if index % 10 == 0 or index == total_variantes - 1:
        print(f"Procesando... ({index + 1}/{total_variantes}) -> Evaluando {cromosoma}:{posicion}")

    try:
        intervalo_base = genome.Interval(chromosome=cromosoma, start=posicion, end=posicion)
        interval = intervalo_base.resize(dna_client.SEQUENCE_LENGTH_1MB)
        variant = genome.Variant(chromosome=cromosoma, position=posicion, reference_bases=ref, alternate_bases=alt)
        
        variante_score_raw = model.score_variant(interval=interval, variant=variant, variant_scorers=scorers_clinvar)
        df_score_limpio = variant_scorers.tidy_scores(variante_score_raw, match_gene_strand=True)
        
        # FILTRO DE TEJIDOS APLICADO AQUÍ
        df_filtrado = df_score_limpio[
            (df_score_limpio['gene_type'] == 'protein_coding') &
            (df_score_limpio['biosample_name'].str.contains('|'.join(tejidos_diana), case=False, na=False))
        ].copy()

        if not df_filtrado.empty:
            df_filtrado.insert(0, 'VAR_ALT', alt)
            df_filtrado.insert(0, 'VAR_REF', ref)
            df_filtrado.insert(0, 'VAR_POS', posicion)
            df_filtrado.insert(0, 'VAR_CHROM', cromosoma_raw.replace('chr', ''))
            
            df_filtrado['abs_diff'] = df_filtrado['raw_score'].abs()
            
            posibles_nombres_columna = ['scorer', 'track', 'feature', 'scorer_name', 'track_name', 'annotation', 'name']
            col_modalidad = next((col for col in posibles_nombres_columna if col in df_filtrado.columns), None)
            
            if not col_modalidad:
                raise KeyError("Columna de modalidad (scorer/track) no encontrada en la respuesta de la API.")
            
            def aplicar_w_m(modalidad_str):
                if 'junction' in str(modalidad_str).lower(): return 0.2
                return 1.0
                
            df_filtrado['w_m'] = df_filtrado[col_modalidad].apply(aplicar_w_m)
            df_filtrado['weighted_score'] = df_filtrado['abs_diff'] * df_filtrado['w_m']
            
            df_modalidades = (
                df_filtrado
                .groupby(['VAR_CHROM', 'VAR_POS', 'VAR_REF', 'VAR_ALT', 'gene_name', 'biosample_name', col_modalidad])
                .agg({'weighted_score': 'max', 'quantile_score': 'max'})
                .reset_index()
            )
            
            df_tejidos = (
                df_modalidades
                .groupby(['VAR_CHROM', 'VAR_POS', 'VAR_REF', 'VAR_ALT', 'gene_name', 'biosample_name'])
                .agg({'weighted_score': 'sum', 'quantile_score': 'max'})
                .reset_index()
                .rename(columns={'weighted_score': 'Composite_Score'})
            )
            
            df_colapsado = (
                df_tejidos
                .sort_values('Composite_Score', ascending=False)
                .drop_duplicates(subset=['VAR_CHROM', 'VAR_POS', 'VAR_REF', 'VAR_ALT', 'gene_name'], keep='first')
            )
            
            if not df_colapsado.empty:
                columnas_finales = [
                    'VAR_CHROM', 'VAR_POS', 'VAR_REF', 'VAR_ALT', 'gene_name', 
                    'Composite_Score', 'quantile_score', 'biosample_name'
                ]
                
                file_exists = os.path.exists(archivo_salida) and os.path.getsize(archivo_salida) > 0
                df_colapsado[columnas_finales].to_csv(
                    archivo_salida, 
                    mode='a', 
                    sep='\t', 
                    index=False, 
                    header=not file_exists 
                )
            
    except Exception as e:
        print(f"Error en variante chr{cromosoma_raw}:{posicion} - {e}")
        continue

print(f"\nConsulta completada. Archivo guardado en: '{archivo_salida}'")

/home/alejandro/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 CONSULTA ALPHAGENOME
Procesando... (1/102) -> Evaluando chr19:15170800
Procesando... (11/102) -> Evaluando chr12:5996214
Procesando... (21/102) -> Evaluando chr17:47292202
Procesando... (31/102) -> Evaluando chr4:154609725
Procesando... (41/102) -> Evaluando chr3:149129855
Procesando... (51/102) -> Evaluando chr16:57661812
Procesando... (61/102) -> Evaluando chr3:149167124
Procesando... (71/102) -> Evaluando chr3:46991482
Procesando... (81/102) -> Evaluando chr13:42199894
Procesando... (91/102) -> Evaluando chr2:43851752
Procesando... (101/102) -> Evaluando chrX:154359007
Procesando... (102/102) -> Evaluando chr7:137846802

Consulta completada. Archivo guardado en: 'resultados_alphagenome/anotacion_alphagenome.tsv'


In [8]:
df_ag = pd.read_csv("resultados_alphagenome/anotacion_alphagenome.tsv", sep='\t')
df_ag.columns

Index(['VAR_CHROM', 'VAR_POS', 'VAR_REF', 'VAR_ALT', 'gene_name',
       'Composite_Score', 'quantile_score', 'biosample_name'],
      dtype='object')

# Filtrado de Variantes

In [9]:
import numpy as np
from IPython.display import display_html

archivo_entrada = 'resultados_alphagenome/anotacion_alphagenome.tsv'
try:
    df_ag = pd.read_csv(archivo_entrada, sep='\t')
except FileNotFoundError:
    print(f"Error, no se encuentra el archivo '{archivo_entrada}'.")
    df_ag = pd.DataFrame()

if not df_ag.empty:
    try:
        with open('panel_genes.txt', 'r') as f:
            mi_panel = [line.strip().upper() for line in f if line.strip()]
        print(f"Panel cargado: {len(mi_panel)} genes en la lista.")
    except FileNotFoundError:
        print(f"Error, no se encuentra el archivo 'panel_genes.txt'.")
        mi_panel = []

    if mi_panel:
        columna_gen = 'gene_name'

        if columna_gen:
            df_ag['gen_match'] = df_ag[columna_gen].astype(str).str.upper()
            df_filtrado = df_ag[df_ag['gen_match'].isin(mi_panel)].copy()
            df_filtrado = df_filtrado.drop(columns=['gen_match']) 
            
            if 'Composite_Score' in df_filtrado.columns:
                df_filtrado['Composite_Score'] = pd.to_numeric(df_filtrado['Composite_Score'], errors='coerce')
                
                # Ordenar asegurando valores absolutos descendentes
                df_filtrado['abs_q'] = df_filtrado['Composite_Score'].abs()
                df_filtrado = df_filtrado.sort_values('abs_q', ascending=False).drop(columns=['abs_q'])
            
            print(f"Filtro aplicado: de {len(df_ag)} anotaciones totales, {len(df_filtrado)} pertenecen al panel.")
            
            display(df_filtrado.head(15))
            
            os.makedirs('resultados_alphagenome', exist_ok=True)
            
            ruta_csv = "resultados_alphagenome/Variantes_AlphaGenome_Filtradas_Panel.csv"
            
            df_filtrado.to_csv(ruta_csv, index=False)
            
        else:
            print("\nNo se encuentra la columna de genes en el archivo TSV.")

Panel cargado: 203 genes en la lista.
Filtro aplicado: de 1491 anotaciones totales, 123 pertenecen al panel.


,VAR_CHROM,VAR_POS,VAR_REF,VAR_ALT,gene_name,Composite_Score,quantile_score,biosample_name
241,4,186280119,CAGGTAAACTGAGAGTTCTGCATTCTGGCTGAGAGTGACCAGCCCC...,C,F11,5.281500,0.999990,liver
1345,7,80661212,T,C,CD36,5.132859,0.999990,liver
1206,4,186284261,G,GGTAGAGTCACCTAAGATTTTGCGTGTCTACAGTGGCATTTTAAAT...,F11,3.604708,0.999990,liver
390,20,35176156,G,GTCTATCCACAGTTCCTCTGACCA,PROCR,3.177027,0.999988,endothelial cell of umbilical vein
1101,4,154612201,C,T,FGG,1.719979,0.999990,liver
1438,2,43824423,G,C,ABCG5,1.190556,0.999963,liver
418,13,42159244,C,CTTTTTTTTT,DGKH,0.973223,0.999868,liver
1162,13,42221318,G,A,DGKH,0.807018,0.999710,endothelial cell of umbilical vein
1449,X,154359007,T,C,FLNA,0.795012,0.999906,endothelial cell of umbilical vein
1166,3,47001055,A,ACGTGG,NBEAL2,0.725661,0.999674,right lobe of liver


In [10]:
len(df_filtrado)

123